# Dry Bean Dataset — Data Cleaning

### Objectives

- Identify missing values in a dataset.
- Count missing observations and missing cells.
- Examine missing-value distributions across features.
- Remove incomplete observations.
- Replace missing values using statistical measures.
- Compare different data cleaning strategies.
- Evaluate how missing data affects descriptive statistics.

Because the original Dry Bean Dataset contains no missing values,
we will introduce artificial missing values into a separate
copy to simulate common data quality problems.

In [162]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_excel(Path("../data/raw/Dry_Bean_Dataset.xlsx"))

## Simulating Missing Data

The original dataset will remain unchanged.

In [163]:
rng = np.random.default_rng(42) # random number generator seed

df_missing = df.copy()

feature_columns = df_missing.select_dtypes(include="number").columns # numerical features

mask_array = rng.random(size=(len(df_missing), len(feature_columns))) < 0.05
missing_mask = pd.DataFrame(mask_array, index=df_missing.index, columns=feature_columns)

df_missing[feature_columns] = df_missing[feature_columns].mask(missing_mask).astype(float)# .loc give opportunity to filter rows but you cannot edit data by .loc()

## Missing Data Analysis

In [164]:
df_missing.loc[:, feature_columns].shape
df_missing.isna().sum().sum()
df_missing.isna().sum()
df_missing[feature_columns].isna().sum() / len(df_missing) * 100

df_missing.loc[df_missing[feature_columns].isna().any(axis=1), feature_columns].shape # isna() creates mask and any(axis = 1) checks if anything is true

df_clean = df_missing.dropna()
print(len(df_missing))
print(len(df_clean))

13611
6053


##  Missing Value Imputation

Removing all observations containing missing values may result
in a significant reduction in dataset size.

An alternative strategy is **imputation**, which replaces missing
values with estimated values.

1. **Mean imputation** — replace missing values using the
   arithmetic mean of each numerical feature.
2. **Median imputation** — replace missing values using the
   median of each numerical feature.
3. **Class-specific median imputation** — replace missing values
   using the median calculated separately for each bean variety.

In [ ]:
mean_val = df_missing[feature_columns].mean()
df_mean = df_missing.fillna(mean_val)
df_mean.count()

median_val = df_missing[feature_columns].median()
df_med = df_missing.fillna(median_val)
df_med.count()

df_class_median = df_missing.copy()
df_class_median[feature_columns] = df_missing[feature_columns].fillna(df_missing.groupby('Class')[feature_columns.tolist()].transform('median'))
df_class_median

len(df_clean)
len(df_med)

datasets = {
    "Original": df,
    "Corrupted": df_missing,
    "Deleted-row": df_clean,
    "Imputed mean": df_mean,
    "Imputed median": df_med,
    "Imputed class median": df_class_median
}

df_comp = {}

for name, dataset in datasets.items():
    df_comp[name] = {
        "Mean": dataset["Area"].mean(),
        "Median": dataset["Area"].median(),
        "Std": dataset["Area"].std(),
    }

df_comp = pd.DataFrame(df_comp).T
print(df_comp)

cleaningMethods = {
    "Deleted-row": df_clean,
    "Imputed mean": df_mean,
    "Imputed median": df_med,
    "Imputed class median": df_class_median
}

df_clean_comp = {}

for name, dataset in cleaningMethods.items():
    df_clean_comp[name]={
        "Relative Std": ((dataset["Area"].std()-df["Area"].std())/ df["Area"].std() * 100)
    }


df_clean_comp = pd.DataFrame.from_dict(df_clean_comp).T  
df_clean_comp

                              Mean   Median           Std
Original              53048.284549  44652.0  29324.095717
Corrupted             53007.945349  44651.0  29309.271246
Deleted-row           52650.660169  44290.0  29108.001392
Imputed mean          53007.945349  45604.0  28594.196933
Imputed median        52605.171405  44651.0  28650.166956
Imputed class median  53061.879950  44615.0  29303.197370


,Relative Std
Deleted-row,-0.736917
Imputed mean,-2.489075
Imputed median,-2.298208
Imputed class median,-0.071267


Mean imputation: Often reduces observed variance because missing values are replaced with the column mean.



Median imputation: May be more robust to extreme values but can also alter the distribution.



Class-specific median imputation: Can preserve differences between known groups better than global imputation, although it uses class labels and is not automatically appropriate for predictive modeling.



Listwise deletion: Reduces the number of observations and may distort the sample when missingness is not completely random.

## Comparison of Data Cleaning Strategies

To evaluate the effects of different missing-value handling
strategies, we compare the original dataset with datasets
obtained through listwise deletion and statistical imputation.

The comparison focuses on:

- Number of retained observations
- Number of remaining missing values
- Mean and median of the `Area` feature
- Standard deviation of `Area`
- Relative change in standard deviation

In [196]:
print("ilość brakujących danych",df_missing.isna().sum().sum())
print("ilość wierszy z brakującymi danymi", df_missing.isna().any(axis=1).sum())

datasets = {
    "Original": df,
    "Corrupted": df_missing,
    "Deleted-row": df_clean,
    "Imputed mean": df_mean,
    "Imputed median": df_med,
    "Imputed class median": df_class_median
}

comparision_tab = {}

originalStd = df['Area'].std()
originalQuantity = df['Class'].value_counts(normalize=True)
for name, dataset in datasets.items():
    comparision_tab[name]={
        'Observations': len(dataset),
        'Retention (%)': len(dataset) / len(df) * 100,
        'Mean Change (%)': ((dataset['Area'].mean() - df['Area'].mean())/ df['Area'].mean() * 100),
        'Median Change (%)': ((dataset['Area'].median() - df['Area'].median())/ df['Area'].median() * 100),
        'AreaStd': dataset['Area'].std(),
        'AreaRelativeStd': (dataset['Area'].std() - originalStd)/originalStd * 100,
        'CV': dataset['Area'].std() / dataset['Area'].mean() * 100,
        'ClassDistributionVariety(TVD)': 0.5 * (originalQuantity - dataset["Class"].value_counts(normalize=True).reindex(originalQuantity.index, fill_value=0)).abs().sum()
    }

comparision_tab = pd.DataFrame(comparision_tab).T

comparision_tab.round(4)

ilość brakujących danych 10828
ilość wierszy z brakującymi danymi 7558


,Observations,Retention (%),Mean Change (%),Median Change (%),AreaStd,AreaRelativeStd,CV,ClassDistributionVariety(TVD)
Original,13611.0,100.0000,0.0000,0.0000,29324.0957,0.0000,55.2781,0.0000
Corrupted,13611.0,100.0000,-0.0760,-0.0022,29309.2712,-0.0506,55.2922,0.0000
Deleted-row,6053.0,44.4714,-0.7496,-0.8107,29108.0014,-0.7369,55.2852,0.0072
Imputed mean,13611.0,100.0000,-0.0760,2.1320,28594.1969,-2.4891,53.9432,0.0000
Imputed median,13611.0,100.0000,-0.8353,-0.0022,28650.1670,-2.2982,54.4626,0.0000
Imputed class median,13611.0,100.0000,0.0256,-0.0829,29303.1974,-0.0713,55.2246,0.0000


## 5. Independent Investigation

### Research Question
How do different missing-value handling strategies affect
the statistical characteristics and class distribution of
the Dry Bean Dataset?

### Methodology

To investigate the effects of missing-data handling,
approximately 5% of the numerical feature values were
randomly replaced with `NaN`.

The original dataset was preserved as a reference.

The following datasets were compared:

1. **Original** — the unmodified dataset.
2. **Corrupted** — the dataset containing artificial missing values.
3. **Deleted-row** — observations containing missing values removed.
4. **Imputed mean** — missing values replaced with feature means.
5. **Imputed median** — missing values replaced with feature medians.
6. **Imputed class median** — missing values replaced with
   feature medians calculated separately for each bean variety.

The comparison was based on the following statistical measures:

- **Observation retention (%)** — percentage of observations
  retained relative to the original dataset.
- **Mean change (%)** — relative difference between the
  original and processed mean of `Area`.
- **Median change (%)** — relative difference between the
  original and processed median of `Area`.
- **Standard deviation change (%)** — relative difference
  in the variability of `Area`.
- **Coefficient of variation (CV)** — relative variability
  of `Area`, expressed as a percentage.
- **Total Variation Distance (TVD)** — a measure of the
  difference between the original and processed class
  distributions.

TVD ranges from 0 to 1, where 0 indicates identical class
distributions and larger values indicate greater differences.

### Results

#### Observation Retention

The original dataset contained 13,611 observations.

After introducing artificial missing values, the dataset
still contained the same number of observations, although
some numerical measurements were missing.

The listwise deletion strategy reduced the number of
observations to **[6053]**, corresponding to a retention
rate of **[~44]%**.

All three imputation strategies preserved the original
number of observations.

#### Changes in Descriptive Statistics

The comparison revealed differences in the statistical
properties of `Area` depending on the cleaning strategy.

The method producing the smallest absolute change in
the mean was **[Imputed Median]**.

The method producing the smallest absolute change in
standard deviation was **[Imputed mean and Corrupted]**.

#### Class Distribution

The differences etween class distributions were evaluated
using Total Variation Distance (TVD).

Listwise deletion produced a TVD of **[0.008]**.

The imputation strategies produced TVD values of
**[0]**.

These results indicate that Class distribution didn't change quietly.

### Interpretation

The analysis demonstrates that different missing-value
handling strategies affect the dataset in different ways.

**Listwise deletion** removes incomplete observations,
reducing the available sample size. Although this approach
avoids introducing estimated values, it may result in
substantial information loss. Changes in class proportions
may also occur due to unequal observation retention.

**Mean imputation** preserves all observations and replaces
missing values with the arithmetic mean of each feature.
However, introducing repeated mean values can reduce the
observed variability of a feature.

**Median imputation** also preserves all observations and
is less sensitive to extreme observations than mean
imputation. Nevertheless, replacing missing measurements
with a fixed median may change the feature's distribution.

**Class-specific median imputation** considers differences
between bean varieties by calculating replacement values
separately for each class. This may preserve class-specific
characteristics more effectively than global imputation.

However, class-specific imputation requires knowledge of
the target class. Consequently, it would not generally
be suitable as a preprocessing step for predicting the
variety of previously unseen beans.

The results should also be interpreted in the context of
the missing-data simulation. Because missing values were
introduced independently and randomly, the experiment
approximates the Missing Completely At Random (MCAR)
mechanism.

### Conclusions

The investigation demonstrates that the choice of
missing-value handling strategy involves trade-offs
between observation retention and preservation of
statistical characteristics.

Listwise deletion substantially reduces the sample size
when missing values are distributed across multiple
features.

Imputation methods preserve the number of observations,
but they can affect descriptive statistics and the
distribution of numerical measurements.

For this experiment, **Imputed Class Median** provided the
most appropriate balance between observation retention
and preservation of the investigated statistical
properties.

This recommendation is based on the results in table .

However, the analysis is limited to artificially generated
missing values and primarily focuses on the `Area` feature.

Further analysis of additional features, correlations,
and feature distributions would be required before
generalizing the conclusions to the entire dataset.
